In [1]:
# في هذه الخلية نقرأ ملف الإكسل ونحمل كل الشيتات في قاموس

import pandas as pd

file_path = "trainData.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)

print("Sheet names:", list(all_sheets.keys()))


Sheet names: ['Gemini', 'ChatGPT', 'Tele', 'Telegram', 'DeepSeek']


In [2]:
# في هذه الخلية نجمع كل الشيتات في جدول واحد ونضيف عمود يوضح مصدر كل صف

dfs = []

for sheet_name, sheet_df in all_sheets.items():
    if sheet_df is None or sheet_df.empty:
        continue

    temp = sheet_df.copy()
    temp["source"] = sheet_name
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Total rows:", df.shape[0])
print("Columns:", list(df.columns))
df.head(8)


Total rows: 2321
Columns: ['نص الاستشارة', 'التصنيف', 'source']


,نص الاستشارة,التصنيف,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,القضايا العقارية,Gemini
5,تم استبعادي من مسابقة وظيفية حكومية رغم انطباق...,القضايا الإدارية,Gemini
6,البنك سحب مبلغ أكبر من القسط الشهري المتفق عليه.,القضايا المالية,Gemini
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,Gemini


In [3]:
# في هذه الخلية نحدد أسماء الأعمدة المستخدمة في الكود
# إذا كانت أسماء أعمدتك مختلفة عدلي المتغيرات هنا فقط

TEXT_COL = "نص الاستشارة"
LABEL_COL = "التصنيف"

print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)


Text column: نص الاستشارة
Label column: التصنيف


In [4]:
# في هذه الخلية ننظف النص بشكل بسيط ونوحد بعض الأحرف
# الهدف تسهيل استخراج الكلمات المفتاحية

import re

def normalize_text_ar(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    text = text.replace("ى","ي").replace("ة","ه")
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

df = df[(df[TEXT_COL] != "") & (df[LABEL_COL] != "")]
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)

df["final_text"] = df[TEXT_COL].apply(normalize_text_ar)

print("Rows after cleaning:", df.shape[0])
df[[TEXT_COL, "final_text", LABEL_COL, "source"]].head(5)


Rows after cleaning: 2321


,نص الاستشارة,final_text,التصنيف,source
0,شريكي سحب سيولة من المؤسسة بدون فواتير وش الحل؟,شريكي سحب سيوله من المؤسسه بدون فواتير وش الحل؟,القضايا التجارية,Gemini
1,صاحب العمل فصلني بدون سابق إنذار ولا أعطاني مك...,صاحب العمل فصلني بدون سابق انذار ولا اعطاني مك...,القضايا العمالية,Gemini
2,تعرضت لابتزاز بصور خاصة من حساب وهمي في سناب.,تعرضت لابتزاز بصور خاصه من حساب وهمي في سناب,القضايا الجنائية,Gemini
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,زوجي هجر البيت ولا يصرف علي العيال من ٤ شهور,القضايا الأسرية,Gemini
4,شريت شقة وطلعت فيها عيوب في السباكة والمالك ير...,شريت شقه وطلعت فيها عيوب في السباكه والمالك ير...,القضايا العقارية,Gemini


In [5]:
# في هذه الخلية نحدد اسم الفئتين كما هي موجودة عندك في البيانات

FAMILY = "القضايا الأسرية"
PERSONAL = "قضايا الأحوال الشخصية"

print("Target labels:", FAMILY, "and", PERSONAL)

# نأخذ فقط الصفوف التابعة لهاتين الفئتين لاستخراج الكلمات المفتاحية
df_fp = df[df[LABEL_COL].isin([FAMILY, PERSONAL])].copy()

print("Rows in the two classes:", df_fp.shape[0])
df_fp[[TEXT_COL, LABEL_COL]].head(5)


Target labels: القضايا الأسرية and قضايا الأحوال الشخصية
Rows in the two classes: 437


,نص الاستشارة,التصنيف
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية
9,كيف أثبت حضانة أطفالي إذا كانت الأم غير مؤهلة؟,قضايا الأحوال الشخصية
15,رفع دعوى خلع والزوج يطالب باسترجاع المهر والهد...,القضايا الأسرية
20,الأم تبي تسافر بالعيال بدون إذن الأب وش النظام؟,القضايا الأسرية


In [6]:
# في هذه الخلية نستخرج كلمات مميزة لكل فئة بناء على بياناتكم
# نستخدم تي اف اي دي اف ونحسب متوسط الوزن لكل كلمة داخل كل فئة ثم نأخذ الأعلى

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1,2),
    min_df=2,
    max_features=20000
)

X = vectorizer.fit_transform(df_fp["final_text"])
terms = np.array(vectorizer.get_feature_names_out())

mask_family = (df_fp[LABEL_COL].values == FAMILY)
mask_personal = (df_fp[LABEL_COL].values == PERSONAL)

X_family = X[mask_family]
X_personal = X[mask_personal]

mean_family = np.asarray(X_family.mean(axis=0)).ravel()
mean_personal = np.asarray(X_personal.mean(axis=0)).ravel()

diff_family = mean_family - mean_personal
diff_personal = mean_personal - mean_family

top_k = 250

family_idx = np.argsort(-diff_family)[:top_k]
personal_idx = np.argsort(-diff_personal)[:top_k]

KW_FAMILY_AUTO = terms[family_idx].tolist()
KW_PERSONAL_AUTO = terms[personal_idx].tolist()

print("Auto keywords count family:", len(KW_FAMILY_AUTO))
print("Auto keywords count personal:", len(KW_PERSONAL_AUTO))

print("Sample family keywords:", KW_FAMILY_AUTO[:30])
print("Sample personal keywords:", KW_PERSONAL_AUTO[:30])


Auto keywords count family: 250
Auto keywords count personal: 250
Sample family keywords: ['بسبب', 'الرجعه', 'لتوثيقها في', 'لتوثيقها', 'النكاح', 'اثبات الرجعه', 'الرجعه لتوثيقها', 'النكاح بسبب', 'فسخ', 'عقد النكاح', 'فسخ عقد', 'عقد', 'كيف', 'رفع دعوي', 'رفع', 'ناجز', 'الطلاق', 'ابي', 'الزوج', 'اجراءات', 'اجراءات الطلاق', 'طليقي', 'اذا كان', 'دعوي', 'للزوج', 'في نظام', 'خلع', 'بسبب عدم', 'هي', 'كان']
Sample personal keywords: ['هل', 'هل يمكن', 'بعد', 'طلاقي اريد', 'يمكن', 'الزام', 'السفر', 'بعد طلاقي', 'طلاقي', 'وفاه', 'الاب', 'الزام الاب', 'اريد', 'وهل', 'اطفالي', 'ذلك', 'بعد وفاه', 'بنفقه', 'يمكن ذلك', 'ان', 'الابناء', 'جواز', 'وفاه زوجي', 'الاب بدفع', 'البيع', 'زوجي اريد', 'يمكن البيع', 'اريد بيع', 'بيع', 'مع اطفالي']


In [7]:
# في هذه الخلية نحذف الكلمات القصيرة جدا والكلمات العامة المتكررة
# الهدف تكون الكلمات أكثر دلالة على التصنيف

COMMON_WORDS = set("""
انا انت انتي هو هي هم هن نحن هذا هذه ذلك تلك في من الى على عن قد لقد لا لم لن ثم او ايضا جدا فقط
حق حقوق عند لدي عندي لي له لها معهم معها معك منك لك لكم
""".split())

def filter_keywords(keywords):
    cleaned = []
    for k in keywords:
        k = k.strip()
        if len(k) < 2:
            continue
        if k in COMMON_WORDS:
            continue
        if len(k.split()) == 1 and k in COMMON_WORDS:
            continue
        cleaned.append(k)
    return cleaned

KW_FAMILY_AUTO = filter_keywords(KW_FAMILY_AUTO)
KW_PERSONAL_AUTO = filter_keywords(KW_PERSONAL_AUTO)

print("Filtered keywords count family:", len(KW_FAMILY_AUTO))
print("Filtered keywords count personal:", len(KW_PERSONAL_AUTO))

print("Sample family keywords:", KW_FAMILY_AUTO[:30])
print("Sample personal keywords:", KW_PERSONAL_AUTO[:30])


Filtered keywords count family: 245
Filtered keywords count personal: 246
Sample family keywords: ['بسبب', 'الرجعه', 'لتوثيقها في', 'لتوثيقها', 'النكاح', 'اثبات الرجعه', 'الرجعه لتوثيقها', 'النكاح بسبب', 'فسخ', 'عقد النكاح', 'فسخ عقد', 'عقد', 'كيف', 'رفع دعوي', 'رفع', 'ناجز', 'الطلاق', 'ابي', 'الزوج', 'اجراءات', 'اجراءات الطلاق', 'طليقي', 'اذا كان', 'دعوي', 'للزوج', 'في نظام', 'خلع', 'بسبب عدم', 'كان', 'نظام ناجز']
Sample personal keywords: ['هل', 'هل يمكن', 'بعد', 'طلاقي اريد', 'يمكن', 'الزام', 'السفر', 'بعد طلاقي', 'طلاقي', 'وفاه', 'الاب', 'الزام الاب', 'اريد', 'وهل', 'اطفالي', 'بعد وفاه', 'بنفقه', 'يمكن ذلك', 'ان', 'الابناء', 'جواز', 'وفاه زوجي', 'الاب بدفع', 'البيع', 'زوجي اريد', 'يمكن البيع', 'اريد بيع', 'بيع', 'مع اطفالي', 'السفر مع']


In [8]:
# في هذه الخلية نبني دالة تصنيف تعتمد على جمع نقاط من الكلمات المفتاحية
# إذا كانت نقاط الأحوال الشخصية أعلى نختارها والعكس صحيح
# إذا تعادلت النقاط نعيد التصنيف إلى الأحوال الشخصية لأنها غالبا الأكثر اتساعا في موضوع الأسرة

def score_by_keywords(text: str, keywords: list) -> int:
    score = 0
    for kw in keywords:
        if kw in text:
            score += 1
    return score

def auto_relabel_family_personal(text: str) -> str:
    t = normalize_text_ar(text)
    s_family = score_by_keywords(t, KW_FAMILY_AUTO)
    s_personal = score_by_keywords(t, KW_PERSONAL_AUTO)

    if s_personal > s_family:
        return PERSONAL
    if s_family > s_personal:
        return FAMILY

    return PERSONAL


In [9]:
# في هذه الخلية نطبق التصحيح على الصفوف الخاصة بالفئتين فقط
# نضيف أعمدة جديدة توضح التصنيف الجديد وعدد الكلمات التي تطابقت لكل فئة

def relabel_with_reason(text: str):
    t = normalize_text_ar(text)
    s_family = score_by_keywords(t, KW_FAMILY_AUTO)
    s_personal = score_by_keywords(t, KW_PERSONAL_AUTO)
    new_label = PERSONAL if s_personal >= s_family else FAMILY
    return new_label, s_family, s_personal

df_corrected = df.copy()

mask_fp_all = df_corrected[LABEL_COL].isin([FAMILY, PERSONAL])

results = df_corrected.loc[mask_fp_all, TEXT_COL].apply(relabel_with_reason)

df_corrected.loc[mask_fp_all, "التصنيف_المصحح_تلقائيا"] = results.apply(lambda x: x[0])
df_corrected.loc[mask_fp_all, "نقاط_اسريه"] = results.apply(lambda x: x[1])
df_corrected.loc[mask_fp_all, "نقاط_احوال_شخصيه"] = results.apply(lambda x: x[2])

df_corrected["التصنيف_النهائي"] = df_corrected[LABEL_COL]
df_corrected.loc[mask_fp_all, "التصنيف_النهائي"] = df_corrected.loc[mask_fp_all, "التصنيف_المصحح_تلقائيا"]

changed = (df_corrected.loc[mask_fp_all, "التصنيف_النهائي"] != df_corrected.loc[mask_fp_all, LABEL_COL]).sum()
total_fp = mask_fp_all.sum()

print("Rows in the two classes:", total_fp)
print("Rows changed automatically:", changed)

df_corrected.loc[mask_fp_all, [TEXT_COL, LABEL_COL, "التصنيف_النهائي", "نقاط_اسريه", "نقاط_احوال_شخصيه"]].head(10)


Rows in the two classes: 437
Rows changed automatically: 56


,نص الاستشارة,التصنيف,التصنيف_النهائي,نقاط_اسريه,نقاط_احوال_شخصيه
3,زوجي هجر البيت ولا يصرف على العيال من ٤ شهور.,القضايا الأسرية,القضايا الأسرية,9.0,3.0
7,أخوي الكبير رافض يوزع ورث أبوي ومستولي على الم...,قضايا الأحوال الشخصية,قضايا الأحوال الشخصية,0.0,1.0
9,كيف أثبت حضانة أطفالي إذا كانت الأم غير مؤهلة؟,قضايا الأحوال الشخصية,القضايا الأسرية,8.0,5.0
15,رفع دعوى خلع والزوج يطالب باسترجاع المهر والهد...,القضايا الأسرية,القضايا الأسرية,6.0,0.0
20,الأم تبي تسافر بالعيال بدون إذن الأب وش النظام؟,القضايا الأسرية,القضايا الأسرية,6.0,5.0
24,توفي والدي وترك عقارات في مكة والرياض كيف نحصرها؟,قضايا الأحوال الشخصية,القضايا الأسرية,2.0,1.0
29,أبي أرفع دعوى عضل لأن الوالد رافض يزوجني بدون ...,قضايا الأحوال الشخصية,القضايا الأسرية,10.0,3.0
33,حقوق الزوجة في السكن المستقل بعيداً عن أهل الزوج.,القضايا الأسرية,القضايا الأسرية,5.0,1.0
39,توزيع تركة متوفى لديه زوجتان وأبناء من كلتيهما.,قضايا الأحوال الشخصية,قضايا الأحوال الشخصية,0.0,2.0
43,إسقاط حضانة الأم لأنها تزوجت من رجل أجنبي.,قضايا الأحوال الشخصية,القضايا الأسرية,4.0,3.0


In [10]:
# في هذه الخلية نحفظ ملف جديد باسم مختلف
# نعيد كتابة كل شيت مع إضافة الأعمدة الجديدة على الصفوف الموجودة فيه

output_path = "trainData_copy_auto_fixed.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        if sheet_df is None or sheet_df.empty:
            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
            continue

        temp = sheet_df.copy()

        if (TEXT_COL in temp.columns) and (LABEL_COL in temp.columns):
            temp["final_text"] = temp[TEXT_COL].apply(normalize_text_ar)

            mask_fp = temp[LABEL_COL].astype(str).isin([FAMILY, PERSONAL])
            if mask_fp.any():
                res = temp.loc[mask_fp, TEXT_COL].apply(relabel_with_reason)
                temp.loc[mask_fp, "التصنيف_المصحح_تلقائيا"] = res.apply(lambda x: x[0])
                temp.loc[mask_fp, "نقاط_اسريه"] = res.apply(lambda x: x[1])
                temp.loc[mask_fp, "نقاط_احوال_شخصيه"] = res.apply(lambda x: x[2])

                temp["التصنيف_النهائي"] = temp[LABEL_COL].astype(str)
                temp.loc[mask_fp, "التصنيف_النهائي"] = temp.loc[mask_fp, "التصنيف_المصحح_تلقائيا"]

        temp.to_excel(writer, sheet_name=sheet_name, index=False)

print("Saved new copy:", output_path)


Saved new copy: trainData_copy_auto_fixed.xlsx


In [11]:
# في هذه الخلية نقيس هل التداخل قل داخل البيانات المصححة
# نستخدم نفس فكرة مقارنة التداخل بين الفئتين

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

df_eval = df_corrected.copy()

df_eval = df_eval.dropna(subset=["final_text", "التصنيف_النهائي"]).reset_index(drop=True)

X_text = df_eval["final_text"].fillna("").astype(str)
y_text = df_eval["التصنيف_النهائي"].astype(str)

X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.2, random_state=42, stratify=y_text
)

y_pred_rule = [auto_relabel_family_personal(x) if y in [FAMILY, PERSONAL] else y for x, y in zip(X_test_text, y_test_text)]

labels = sorted(y_test_text.unique())
cm = confusion_matrix(y_test_text, y_pred_rule, labels=labels)

if FAMILY in labels and PERSONAL in labels:
    fam_i = labels.index(FAMILY)
    per_i = labels.index(PERSONAL)
    print("Family to Personal:", cm[fam_i][per_i])
    print("Personal to Family:", cm[per_i][fam_i])
else:
    print("One of the labels was not found in the test split.")


Family to Personal: 0
Personal to Family: 0
